# Day 04 — Decorators: `@tool` & Custom Wrappers

**Phase 1 · Module 1: Tool Calling** · ~30 min

### What I practice today
- [ ] Write a **`@retry`** decorator for tool functions — manual, no library
- [ ] Study how **`@tool`** from LangChain works internally (build a mini version)
- [ ] Add **`@functools.wraps`** to preserve the wrapped function's name & docstring

> Today's Python topic feeds directly into the LangGraph notebook: `@tool` is *the* way you turn a plain function into something an agent can call.

## 1. A decorator is a function that takes a function and returns a function

That's the whole idea. A tool call over the network can fail; you want to wrap the tool with *retry* logic **without editing the tool's body**. A decorator lets you add behaviour *around* a function.

Start with the wrapping done **by hand**, no `@` yet — so the magic is obvious.

In [1]:
def add(a: int, b: int) -> int:
    return a + b

# A "decorator": takes a function, returns a NEW function that wraps it
def with_logging(func):
    def wrapper(*args, **kwargs):
        print(f"-> calling {func.__name__}{args}")
        result = func(*args, **kwargs)
        print(f"<- {func.__name__} returned {result}")
        return result
    return wrapper

# Wrap manually: reassign the name to the wrapped version
add = with_logging(add)
add(2, 3)

-> calling add(2, 3)
<- add returned 5


5

☝️ `*args, **kwargs` let the wrapper accept **any** signature and pass it straight through — so one decorator works on any function. `add = with_logging(add)` is the manual step the `@` syntax will hide next.

## 2. The `@` syntax is exactly that — sugar

`@with_logging` on the line above `def` means *"after defining this function, replace it with `with_logging(func)`"*. Same result, less noise.

In [2]:
def with_logging(func):
    def wrapper(*args, **kwargs):
        print(f"-> calling {func.__name__}{args}")
        result = func(*args, **kwargs)
        print(f"<- returned {result}")
        return result
    return wrapper

@with_logging               # <- identical to  multiply = with_logging(multiply)
def multiply(a: int, b: int) -> int:
    return a * b

multiply(4, 5)

-> calling multiply(4, 5)
<- returned 20


20

## 3. The problem: the wrapper **steals the function's identity**

Look at `multiply` now — its name and docstring are gone, replaced by the wrapper's. For LangChain this is fatal: `@tool` reads the function's **name** and **docstring** to tell the LLM what the tool does. If those get clobbered, the model sees a tool called `wrapper` with no description.

In [3]:
def bad_decorator(func):
    def wrapper(*args, **kwargs):
        """I am the wrapper's docstring."""
        return func(*args, **kwargs)
    return wrapper

@bad_decorator
def area(radius: float) -> float:
    """Return the area of a circle with the given radius."""
    return 3.14159 * radius ** 2

print("name    :", area.__name__)   # 'wrapper'  <- identity lost
print("docstring:", area.__doc__)   # the wrapper's doc, not area's

name    : wrapper
docstring: I am the wrapper's docstring.


## 4. The fix: `@functools.wraps`

`functools.wraps(func)` copies `func`'s metadata (`__name__`, `__doc__`, `__module__`, ...) onto the wrapper. One line, and the wrapped function keeps its identity.

In [4]:
import functools

def good_decorator(func):
    @functools.wraps(func)              # <- copy area's identity onto wrapper
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@good_decorator
def area(radius: float) -> float:
    """Return the area of a circle with the given radius."""
    return 3.14159 * radius ** 2

print("name    :", area.__name__)   # 'area'  ✅
print("docstring:", area.__doc__)   # area's own doc  ✅

name    : area
docstring: Return the area of a circle with the given radius.


☝️ **Rule:** every decorator you write gets `@functools.wraps(func)`. It's free and it prevents the exact class of bug that would make an agent's tool description disappear.

## 5. Decorators that take arguments — one extra layer

`@retry(times=3)` needs to *accept a parameter*. That means one more nested function: the outer call `retry(times=3)` **returns the actual decorator**. Three layers:

```
retry(times=3)      -> returns a decorator
   decorator(func)  -> returns wrapper
      wrapper(...)   -> the retry loop
```

In [5]:
import functools

def retry(times: int = 3):
    """Decorator FACTORY: retry(times=3) returns the real decorator."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(1, times + 1):
                print(f"   attempt {attempt}/{times}")
                # (real retry loop lands in section 6)
                return func(*args, **kwargs)
        return wrapper
    return decorator

@retry(times=2)
def ping() -> str:
    return "pong"

ping()

   attempt 1/2


'pong'

## 6. Build the deliverable — a real `@retry` for flaky tool functions

Tools call networks; networks fail intermittently. A production `@retry` should:
1. catch exceptions, retry up to `times`,
2. **back off** between attempts (wait longer each time),
3. re-raise the last exception if all attempts fail,
4. keep the tool's identity via `@functools.wraps`.

No library — just `functools` and `time`.

In [6]:
import functools, time, random

def retry(times: int = 3, delay: float = 0.1, backoff: float = 2.0):
    """Retry a function on exception, with exponential backoff.

    times   -- max attempts
    delay   -- seconds to wait after the first failure
    backoff -- multiply the delay by this each further failure
    """
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            wait = delay
            last_exc = None
            for attempt in range(1, times + 1):
                try:
                    return func(*args, **kwargs)          # success -> return now
                except Exception as exc:
                    last_exc = exc
                    print(f"   [{func.__name__}] attempt {attempt} failed: {exc}")
                    if attempt < times:
                        time.sleep(wait)
                        wait *= backoff                   # exponential backoff
            raise last_exc                                # all attempts exhausted
        return wrapper
    return decorator

In [7]:
# A flaky "tool" that fails the first 2 calls, then succeeds
_calls = {"n": 0}

@retry(times=4, delay=0.05)
def web_search(query: str) -> str:
    """Search the web and return the top result."""
    _calls["n"] += 1
    if _calls["n"] < 3:
        raise ConnectionError("transient network error")
    return f"top result for {query!r}"

print("RESULT:", web_search("langgraph tools"))
print("identity preserved ->", web_search.__name__, "|", web_search.__doc__)

   [web_search] attempt 1 failed: transient network error
   [web_search] attempt 2 failed: transient network error
RESULT: top result for 'langgraph tools'
identity preserved -> web_search | Search the web and return the top result.


In [8]:
# And the failure path: a tool that ALWAYS fails re-raises after N attempts
@retry(times=3, delay=0.02)
def always_down() -> str:
    raise TimeoutError("service unavailable")

try:
    always_down()
except TimeoutError as e:
    print("re-raised after 3 attempts ->", type(e).__name__, ":", e)

   [always_down] attempt 1 failed: service unavailable
   [always_down] attempt 2 failed: service unavailable
   [always_down] attempt 3 failed: service unavailable
re-raised after 3 attempts -> TimeoutError : service unavailable


☝️ This `@retry` is production-shaped: bounded attempts, exponential backoff, the original exception re-raised (not swallowed), and identity preserved. Drop it on any LangChain tool function and its calls become resilient.

## 7. How LangChain's `@tool` works internally

`@tool` doesn't just wrap the function — it **inspects** it and returns a `StructuredTool` object carrying:
- **`name`** ← the function's `__name__`
- **`description`** ← the function's **docstring** (this is why the docstring matters!)
- **`args_schema`** ← a Pydantic model built from the **type hints**
- **`func`** ← the original callable, reachable via `.invoke()`

Build a stripped-down version to *see* the mechanism. It reads the same three things `@tool` reads: name, docstring, hints.

In [9]:
import functools, inspect
from pydantic import create_model

class MiniTool:
    """A tiny stand-in for LangChain's StructuredTool."""
    def __init__(self, func):
        functools.wraps(func)(self)          # copy name/doc onto the tool
        self.name = func.__name__
        self.description = (func.__doc__ or "").strip()
        # Build an args schema from the type hints (like @tool does)
        sig = inspect.signature(func)
        fields = {p.name: (p.annotation, ...) for p in sig.parameters.values()}
        self.args_schema = create_model(f"{func.__name__}_args", **fields)
        self._func = func

    def invoke(self, arguments: dict):
        validated = self.args_schema(**arguments)     # validate inputs
        return self._func(**validated.model_dump())

def mini_tool(func):
    """Our @tool: turn a plain function into a MiniTool."""
    return MiniTool(func)

In [10]:
@mini_tool
def calculator(a: float, b: float) -> float:
    """Add two numbers together and return the sum."""
    return a + b

print("name       :", calculator.name)
print("description:", calculator.description)
print("args schema:", calculator.args_schema.model_json_schema()["properties"])

# The LLM would emit a JSON blob like {'a': 2, 'b': 3}; the tool validates + runs it
print("invoke     :", calculator.invoke({"a": 2, "b": 3}))

name       : calculator
description: Add two numbers together and return the sum.
args schema: {'a': {'title': 'A', 'type': 'number'}, 'b': {'title': 'B', 'type': 'number'}}
invoke     : 5.0


☝️ That is the essence of LangChain's `@tool`: **name + docstring + type hints → a self-describing, validated callable** the LLM can pick and call. Now the Day-1 lessons click together — the docstring *is the tool's description*, and the type hints *are the tool's input contract*. Tomorrow's LangGraph notebook uses the real `@tool` for exactly this.

## 8. Recap + checklist

| Tool | Why it was used here |
|------|----------------------|
| **decorator** | wrap a function to add behaviour (logging, retry) without editing its body |
| **`@` syntax** | sugar for `f = decorator(f)` |
| **`*args, **kwargs`** | let one wrapper accept any signature |
| **`@functools.wraps`** | copy `__name__`/`__doc__` so the wrapped fn keeps its identity — critical for `@tool` |
| **decorator factory** | extra layer so a decorator can take arguments, e.g. `@retry(times=3)` |
| **`@retry`** | bounded attempts + exponential backoff + re-raise → resilient tool calls |
| **`@tool` internals** | reads name + docstring + hints → a self-describing, validated callable |

